In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:
# k = 0
k = 0.5

folderVec = [
    #! k=0
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k0_32dx/probes/probe2",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k0_40dx/probes/probe2",


    #! k=0.5
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx/probes/probe2",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_noGeoUpdate/probes/probe2",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_noGeoUpdate/probes/probe2",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_larger_noGeoUpdate/probes/probe2",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_wider_L42_noGeoUpdate/probes/probe2"
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/cml_3d_u001_k05_32dx_noGeoUpdate/probes/probe2",

    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_larger_noGeoUpdate/probes/probe2",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_40dx_larger_noGeoUpdate/probes/probe2",

]

rho0 = 1.0
cs2 = 1.0 / 3.0  # lattice speed of sound squared

U0Vec = [
    0.01,
    0.01,
    0.01,
    0.01,
    0.01,
]

startStep = 20000

avgStepRg = [
    [100000, 200000],
    [100000, 200000],
    [100000, 200000],
    [100000, 200000],
]

colorVec = [
    "#05A361",
    "#D33737",
    # "#001AFF",
    # '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    # 's',
    "d",
    'P',
    '*'
]

labelVec = [
    "IBB+MovBody D32 L10 Y10",
    "IBB D32 L10 Y10",
    # "IBB D32 L10 Y16",
    "IBB D32 L25 Y16",
    "IBB D32 L42 Y16",
    # "Present LBM IBB D32 L25",
    # "IBB D40 L25",

    # "Present LBM BGK D/dx=32 k=0.5",
    # "Present LBM BGK D/dx=32 noGeoUpdate",
    # "Present LBM Cumulant D/dx=32 noGeoUpdate",
]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec


In [ ]:

thetaVecVec = []
pCoefVecVec = []
for case in range(0, len(folderVec)):
    files = sorted(glob.glob(os.path.join(folderVec[case], "probe*.txt")))
    nProbes = len(files)

    stepCol = read_col_probe(files[0], 0, 2)
    idxStart = int(np.abs(stepCol - avgStepRg[case][0]).argmin())
    idxEnd = int(np.abs(stepCol - avgStepRg[case][1]).argmin())
    thetaVec = []
    pCoefVec = []
    dynPressure = 0.5 * rho0 * U0Vec[case]**2
    for iProbe in range(0, len(files)):
        x, y, z = get_probe_coords(files[iProbe])
        theta = np.degrees(np.arctan2(y, x)) % 360
        thetaVec.append(theta)
        rhoCol = read_col_probe(files[iProbe], 2, 2)
        pressure = (rhoCol - rho0) * cs2
        avgPressure = np.mean(pressure[idxStart:idxEnd])
        pCoef = avgPressure / dynPressure
        pCoefVec.append(pCoef)
    thetaVecVec.append(thetaVec)
    pCoefVecVec.append(pCoefVec)
    

In [ ]:
refCpFile1 = "../ref/cp_fallah_k0_n1.csv"
refTheta1 = read_csv_col(refCpFile1, 0, 1)
refCp1 = read_csv_col(refCpFile1, 1, 1)

refCpFile2 = "../ref/cp_fallah_k05_n1.csv"
refTheta2 = read_csv_col(refCpFile2, 0, 1)
refCp2 = read_csv_col(refCpFile2, 1, 1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6), facecolor='w', edgecolor='w')

for case in range(0, len(folderVec)):
    ax.plot(thetaVecVec[case], pCoefVecVec[case], c=colorVec[case], lw=0, marker=markerVec[case], fillstyle='none', ms=6, mew=1.0, markevery=3, label=labelVec[case])
    # ax.plot(thetaVecVec[case][1:], pCoefVecVec[case][1:], c=colorVec[case], lw=1.5, fillstyle='none', label=labelVec[case])
# ax.plot(refTheta1, refCp1, lw=1.5, c='k', label='Fallah et al., LBM MRT k=0')

if k == 0:
    ax.plot(refTheta1, refCp1, c='k', linestyle='-', lw=1.5, label='Fallah et al., LBM MRT k=0')
elif k == 0.5:
    ax.plot(refTheta2, refCp2, c='k', linestyle='--', lw=1.5, label='Fallah et al., LBM MRT k=0.5')

ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$C_p$")
ax.set_xlim(0, 360)
ax.set_ylim(-4, 3)
ax.set_xticks(np.arange(0, 361, 60, dtype=int))
ax.tick_params(width=2.0, axis='both', direction='in')
for spine in ax.spines.values():
    spine.set_linewidth(2.0)

ax.legend(loc='lower left', frameon=False, fontsize=13)